# Combining Datasets with Pandas

Real-world data rarely arrives in a single, tidy table. You often need to bring together information from multiple sources — joining customer records with order data, appending new rows from a fresh batch, or adding a computed column from a separate operation. Pandas provides several tools for this.

The three main functions are `pd.merge()`, `df.join()`, and `pd.concat()`. They all combine DataFrames, but they differ in their **defaults** and the types of combination they make easiest. Understanding when to use each one will save you a lot of debugging time.

This notebook also introduces **binning** with `pd.cut()` — a technique for grouping a continuous column into labelled categories — and **pivot tables** with `pd.pivot_table()`, which lets you reshape and summarise a DataFrame in one step.

## Learning Objectives

By the end of this notebook you will be able to:

1. Combine DataFrames side-by-side using `pd.concat()` with `axis=1`

2. Combine DataFrames by row using `pd.concat()` with `axis=0`

3. Join DataFrames on their index using `df.join()`

4. Merge DataFrames on a shared column using `pd.merge()`

5. Explain the four join types: `left`, `right`, `inner`, and `outer`

6. Bin a continuous column into categories using `pd.cut()`

7. Create a summary pivot table using `pd.pivot_table()`

---

## Setup

In [ ]:
# Import pandas and load the red wine dataset
import pandas as pd

wine_df = pd.read_csv('../data/winequality-red.csv', delimiter=';')
wine_df.head()

In [ ]:
# Check the unique quality scores in the dataset
wine_df['quality'].unique()

---

## Creating Dummy Variables with `pd.get_dummies()`

Before looking at how to combine DataFrames, let's create a second one to work with. `pd.get_dummies()` converts a **categorical column** into a set of **binary (0/1) indicator columns** — one per category. This is a common step in preparing data for machine learning.

We will use `quality` (which has values 3–8) to produce columns like `quality_3`, `quality_4`, etc.

In [ ]:
# Convert the quality column into dummy/indicator variables
# pd.get_dummies returns a new DataFrame — one column per category
quality_dummies = pd.get_dummies(wine_df['quality'], prefix='quality')
quality_dummies.head()

---

## `df.join()` — Combining on Index

`df.join()` attaches another DataFrame (or Series) to the current one, **aligning on the index** by default. It is the fastest way to add new columns when the indices already match — which is the case here, since both `wine_df` and `quality_dummies` share the default integer index.

Think of it like adding extra columns to a spreadsheet where both sheets share the same row numbers.

In [ ]:
# Join the dummy columns onto the original wine DataFrame, aligned by index
joined_df = wine_df.join(quality_dummies)
joined_df.head()

### Join Types

The `how` parameter controls which rows to keep when the two DataFrames don't share all the same indices:

| `how` value | SQL equivalent | Keeps... |
|---|---|---|
| `'left'` | LEFT OUTER JOIN | All rows from the left DataFrame |
| `'right'` | RIGHT OUTER JOIN | All rows from the right DataFrame |
| `'outer'` | FULL OUTER JOIN | All rows from both DataFrames |
| `'inner'` | INNER JOIN | Only rows present in **both** DataFrames |

The default is `'left'`. Missing values in the joined result appear as `NaN`.

![Join Methods](../images/join_types.png)

---

## `pd.concat()` — Stacking DataFrames

`pd.concat()` stacks a **list** of DataFrames together. The direction depends on the `axis` parameter:

- `axis=0` (default): stacks **rows** — appends one DataFrame below another
- `axis=1`: stacks **columns** — places DataFrames side by side

**Concat with `axis=0`** (stacking rows):

![Concat Axis 0](../images/concat_axis_0.png)

---

**Concat with `axis=1`** (stacking columns):

![Concat Axis 1](../images/concat_axis_1.png)

In [ ]:
# Combine side-by-side with axis=1 — same result as .join() here
# Both DataFrames share the same index, so alignment is automatic
joined_df2 = pd.concat([wine_df, quality_dummies], axis=1)
joined_df2.head()

---

## `pd.merge()` — Combining on a Shared Column

`pd.merge()` joins two DataFrames based on **matching values in one or more columns** (rather than the index). This is equivalent to a SQL `JOIN`.

It is the best tool when:
- The two DataFrames share a **common column** (like a customer ID or date)
- The rows are not necessarily in the same order
- You want to match records precisely by value

In [ ]:
# Load both red and white wine datasets for a cross-dataset comparison
red_wines_df = pd.read_csv('../data/winequality-red.csv', delimiter=';')
white_wines_df = pd.read_csv('../data/winequality-white.csv', delimiter=';')

In [ ]:
# Confirm both datasets share the same column names
print('Red columns: ', list(red_wines_df.columns))
print('White columns:', list(white_wines_df.columns))

In [ ]:
# Create a summary: average fixed acidity per quality score for each wine type
# Pandas 3.0 note: numeric_only=True prevents a warning on non-numeric columns
red_quality_df = red_wines_df.groupby('quality')['fixed acidity'].mean().reset_index()
white_quality_df = white_wines_df.groupby('quality')['fixed acidity'].mean().reset_index()

red_quality_df.head()

In [ ]:
# Merge the two summary tables on the shared 'quality' column
# The suffixes parameter disambiguates the two 'fixed acidity' columns
merged_df = pd.merge(
    red_quality_df,
    white_quality_df,
    on=['quality'],
    suffixes=[' red', ' white']
)
merged_df

Notice that only quality scores present in **both** datasets appear. By default, `pd.merge()` performs an **inner join**. Use `how='outer'` to keep all rows.

In [ ]:
# Achieve the same result using pd.concat() with axis='columns'
# join='inner' keeps only quality scores present in both DataFrames
concat_result = pd.concat(
    [red_quality_df.set_index('quality'), white_quality_df.set_index('quality')],
    axis='columns',
    join='inner'
)
concat_result.columns = ['fixed acidity red', 'fixed acidity white']
concat_result

---

## Binning with `pd.cut()`

Sometimes a continuous column is more useful when divided into labelled **bins** or categories. `pd.cut()` does this automatically.

Pass a Series and the number of bins. Pandas will divide the value range into equal-width intervals and return a **Categorical Series** with one label per row.

In [ ]:
# Divide fixed acidity into 5 equal-width bins
# Each element gets assigned to a bin — the output is a Categorical Series
fa_bins = pd.cut(red_wines_df['fixed acidity'], bins=5)
fa_bins.head(10)

In [ ]:
# Count how many rows fall into each bin
fa_bins.value_counts().sort_index()

In [ ]:
# Attach the bin column to the original DataFrame using pd.concat()
# Pandas 3.0 note: use pd.concat() instead of the removed .append() method
red_wines_df = pd.concat(
    [red_wines_df, fa_bins.rename('acidity_bin')],
    axis=1
)
red_wines_df.head()

---

## Pivot Tables with `pd.pivot_table()`

A **pivot table** is a powerful way to summarise data across two dimensions at once. It is similar to a `groupby`, but the result is a 2D table rather than a flat Series.

The key parameters are:
- `values` — the column to aggregate
- `index` — which column becomes the row labels
- `columns` — which column becomes the column headers
- `aggfunc` — how to aggregate (e.g. `'mean'`, `'count'`, `'sum'`)

In [ ]:
# Pivot table: average alcohol content by quality score and acidity bin
# index = rows, columns = column headers, values = what to average
pivot = pd.pivot_table(
    red_wines_df,
    values='alcohol',
    index='quality',
    columns='acidity_bin',
    aggfunc='mean'
)
pivot

Each cell shows the average `alcohol` content for wines in that combination of quality score (row) and acidity bin (column). `NaN` appears where no wines fall into that combination.

---

## Practice Challenges

### Challenge 1: Combine, then merge

Join `df1` and `df2` along rows (stack them vertically into one DataFrame), then merge the result with `df3` on the shared `student_id` column.

After you have the merged result, look carefully at the rows: do you notice anything unusual? Can you explain why it happened?

In [ ]:
# Starter data — run this cell first
df1 = pd.DataFrame({
    'student_id': ['S1', 'S2', 'S3', 'S4', 'S5'],
    'name': ['Erika Raaf', 'Nadja Berens', 'Florentin Kleist', 'Dorothea Eibl', 'Gerhard Bihlmeier'],
    'subject': ['Math', 'Biology', 'Biology', 'English', 'Philosophy']
})
df2 = pd.DataFrame({
    'student_id': ['S6', 'S7', 'S8', 'S9', 'S10'],
    'name': ['Franz Xaver Schild', 'Ann-Kathrin Heiny', 'Jens Hüls', 'Vera Kagan', 'Paula Brodersen'],
    'subject': ['Chemistry', 'Economics', 'Math', 'Math', 'Social Science']
})
df3 = pd.DataFrame({
    'student_id': ['S1', 'S2', 'S3', 'S4', 'S5', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 'S13'],
    'marks': [23, 45, 12, 67, 21, 55, 33, 14, 56, 83, 88, 12]
})

In [ ]:
# YOUR CODE HERE

### Challenge 2: Weather data join

You have two weather datasets: one with monthly mean temperatures and one with monthly maximum temperatures (available for only 4 months). Combine them into a single DataFrame **without losing any rows from either source**.

Bonus: after joining, can you fill the missing values in `Max TemperatureF` with the overall average maximum temperature?

In [ ]:
# Starter data — run this cell first
weather_mean_data = {
    'Mean TemperatureF': [53.1, 70., 34.93, 28.71, 32.35, 72.87, 70.13, 35., 62.61, 39.8, 55.45, 63.77],
    'Month': ['Apr', 'Aug', 'Dec', 'Feb', 'Jan', 'Jul', 'Jun', 'Mar', 'May', 'Nov', 'Oct', 'Sep']
}
weather_max_data = {
    'Max TemperatureF': [68, 89, 91, 84],
    'Month': ['Jan', 'Apr', 'Jul', 'Oct']
}

mean_df = pd.DataFrame(weather_mean_data)
max_df = pd.DataFrame(weather_max_data)

In [ ]:
# YOUR CODE HERE

---

## Key Takeaways

- Use **`df.join()`** to attach columns from another DataFrame or Series — it aligns on the **index** by default

- Use **`pd.concat()`** with `axis=0` to stack rows, and `axis=1` to stack columns

- Use **`pd.merge()`** to join on a **shared column** (like a key or ID), similar to SQL `JOIN`

- The `how` parameter (`'left'`, `'right'`, `'inner'`, `'outer'`) controls which rows are kept when indices or keys don't fully overlap

- Use **`pd.cut()`** to bin a continuous column into labelled categorical intervals

- Use **`pd.pivot_table()`** to summarise data across two dimensions at once

- `NaN` values in a merged or pivoted result signal rows that had no matching key in one of the sources


---

## Check Your Understanding

1. What is the key difference between `df.join()` and `pd.merge()`? When would you choose one over the other?

2. You call `pd.concat([df_a, df_b])` and the result has duplicate index values. What caused this, and how would you fix it?

3. An `inner` merge of two DataFrames returns fewer rows than either original. Why?

4. You use `pd.cut(df['price'], bins=4)`. What does "equal-width" mean here, and how is it different from equal-count bins?

5. In a pivot table, some cells contain `NaN`. What does that tell you about the data?

### **Practice in:** [The Final Practice](The_Last_Pandas_Practice.ipynb)